# Course 13: MLOps for Generative AI
**GCP Machine Learning Engineer Certification - Section 5**

This notebook covers hands-on exercises for GenAI MLOps on Vertex AI:
1. Vertex AI Supervised Fine-Tuning Job Setup
2. Prompt Versioning Pattern
3. Automated Evaluation: LLM-as-Judge with Gemini
4. RAG Pipeline: Embed, Retrieve, Generate
5. Cost Tracking for GenAI API Calls
6. Model Comparison Framework

> **Note:** Fine-tuning cells are commented out to avoid accidental costs. Uncomment and run only when ready.

---
## 0. Setup & Dependencies

In [ ]:
# Install required packages (uncomment if needed)
# !pip install google-cloud-aiplatform vertexai pyyaml pandas numpy --quiet

In [ ]:
import json
import yaml
import time
import hashlib
from datetime import datetime
from typing import Dict, List, Any, Optional

# Google Cloud (uncomment when running on GCP)
# import vertexai
# from google.cloud import aiplatform
# from vertexai.generative_models import GenerativeModel, GenerationConfig
# from vertexai.tuning import sft

print("Imports loaded successfully.")

In [ ]:
# Configuration - UPDATE THESE for your project
PROJECT_ID = "your-gcp-project-id"
LOCATION = "us-central1"
BUCKET_NAME = "your-training-bucket"

# Uncomment to initialize Vertex AI
# vertexai.init(project=PROJECT_ID, location=LOCATION)

---
## 1. Vertex AI Supervised Fine-Tuning Job Setup

This section shows how to prepare training data and launch a supervised fine-tuning (SFT) job on Vertex AI.

**WARNING:** Running the actual fine-tuning job incurs real costs ($2-8+ per 1K training examples). The code below is fully functional but commented out.

In [ ]:
# Step 1: Prepare training data in the required JSONL format

training_examples = [
    {
        "messages": [
            {"role": "system", "content": "You are a customer support agent for a cloud computing company. Be concise, helpful, and professional."},
            {"role": "user", "content": "How do I create a new Cloud Storage bucket?"},
            {"role": "model", "content": "To create a Cloud Storage bucket:\n1. Go to console.cloud.google.com/storage\n2. Click 'Create Bucket'\n3. Enter a globally unique name\n4. Choose location type and region\n5. Select storage class (Standard for frequent access)\n6. Set access control (Uniform recommended)\n7. Click 'Create'\n\nAlternatively via CLI: `gsutil mb gs://your-bucket-name`"}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a customer support agent for a cloud computing company. Be concise, helpful, and professional."},
            {"role": "user", "content": "My VM is running slowly. What should I check?"},
            {"role": "model", "content": "Check these common causes for slow VM performance:\n1. **CPU utilization** - Check in Monitoring; upgrade machine type if consistently >80%\n2. **Memory pressure** - Look for swap usage; consider more RAM\n3. **Disk I/O** - Switch from standard to SSD persistent disks\n4. **Network bandwidth** - Larger machine types get more bandwidth\n5. **Process issues** - SSH in and run `top` to find resource-hogging processes\n\nQuick fix: Try stopping and restarting the VM on a different host with `gcloud compute instances move`."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a customer support agent for a cloud computing company. Be concise, helpful, and professional."},
            {"role": "user", "content": "What's the difference between Cloud Run and GKE?"},
            {"role": "model", "content": "**Cloud Run** - Fully managed serverless container platform. Best for stateless HTTP services. Scales to zero, pay per request. No cluster management.\n\n**GKE (Google Kubernetes Engine)** - Managed Kubernetes. Best for complex microservice architectures needing fine-grained control, stateful workloads, or GPU access. You manage node pools and cluster config.\n\n**Choose Cloud Run** if you want simplicity and auto-scaling. **Choose GKE** if you need Kubernetes features, custom networking, or persistent storage."}
        ]
    }
]

# Write to JSONL file
with open("train.jsonl", "w") as f:
    for example in training_examples:
        f.write(json.dumps(example) + "\n")

print(f"Created train.jsonl with {len(training_examples)} examples")
print("\nSample entry:")
print(json.dumps(training_examples[0], indent=2))

In [ ]:
# Step 2: Upload to Cloud Storage and launch fine-tuning job
# *** COMMENTED OUT TO AVOID COSTS ***

# # Upload training data to GCS
# !gsutil cp train.jsonl gs://{BUCKET_NAME}/fine-tuning/train.jsonl

# # Launch supervised fine-tuning job
# from vertexai.tuning import sft
#
# tuning_job = sft.train(
#     source_model="gemini-2.0-flash-001",
#     train_dataset=f"gs://{BUCKET_NAME}/fine-tuning/train.jsonl",
#     # validation_dataset=f"gs://{BUCKET_NAME}/fine-tuning/val.jsonl",  # Optional
#     epochs=3,
#     adapter_size=4,               # LoRA rank: 1, 4, 8, 16
#     learning_rate_multiplier=1.0,  # Scale the default LR
#     tuned_model_display_name="customer-support-v1",
# )
#
# # Monitor the job
# print(f"Job state: {tuning_job.state}")
# print(f"Job name: {tuning_job.resource_name}")
#
# # Wait for completion (can take 30min - several hours)
# # tuning_job.wait()
# # print(f"Tuned model endpoint: {tuning_job.tuned_model_endpoint_name}")

print("Fine-tuning code is ready but commented out to avoid costs.")
print("Uncomment and run when you're ready to fine-tune.")
print("\nEstimated cost for 3 examples: ~$0.01")
print("Estimated cost for 500 examples: ~$2-5")

---
## 2. Prompt Versioning Pattern

A production prompt management system using Python dicts and YAML for version control.

In [ ]:
class PromptRegistry:
    """A simple prompt versioning and management system.
    
    In production, this would be backed by a database or config service.
    For exam purposes, understand the concept of versioning prompts.
    """
    
    def __init__(self):
        self.prompts: Dict[str, Dict] = {}
        self.active_versions: Dict[str, str] = {}  # task -> version_id
    
    def register(self, task: str, version: str, template: str, 
                 model: str, temperature: float = 0.7,
                 metadata: Optional[Dict] = None) -> str:
        """Register a new prompt version."""
        version_id = f"{task}_v{version}"
        self.prompts[version_id] = {
            "task": task,
            "version": version,
            "template": template,
            "model": model,
            "temperature": temperature,
            "metadata": metadata or {},
            "created_at": datetime.now().isoformat(),
            "content_hash": hashlib.sha256(template.encode()).hexdigest()[:12],
        }
        return version_id
    
    def activate(self, version_id: str):
        """Set a prompt version as the active production version."""
        if version_id not in self.prompts:
            raise ValueError(f"Version {version_id} not found")
        task = self.prompts[version_id]["task"]
        self.active_versions[task] = version_id
        print(f"Activated {version_id} for task '{task}'")
    
    def get_active(self, task: str) -> Dict:
        """Get the active prompt for a task."""
        version_id = self.active_versions.get(task)
        if not version_id:
            raise ValueError(f"No active version for task '{task}'")
        return self.prompts[version_id]
    
    def render(self, task: str, **variables) -> str:
        """Render the active prompt template with variables."""
        prompt_config = self.get_active(task)
        return prompt_config["template"].format(**variables)
    
    def list_versions(self, task: str) -> List[Dict]:
        """List all versions for a task."""
        return [
            {"id": vid, **v}
            for vid, v in self.prompts.items()
            if v["task"] == task
        ]
    
    def export_yaml(self, filepath: str = "prompts.yaml"):
        """Export all prompts to YAML for version control."""
        export = {
            "active_versions": self.active_versions,
            "prompts": self.prompts,
        }
        with open(filepath, "w") as f:
            yaml.dump(export, f, default_flow_style=False, sort_keys=False)
        print(f"Exported {len(self.prompts)} prompts to {filepath}")


# Create and populate the registry
registry = PromptRegistry()

# Register prompt versions
registry.register(
    task="summarize",
    version="1.0",
    template="Summarize the following text in {num_sentences} sentences:\n\n{text}",
    model="gemini-2.0-flash",
    temperature=0.3,
    metadata={"eval_score": 0.82, "author": "team-a"}
)

registry.register(
    task="summarize",
    version="2.0",
    template="""You are a technical writer creating executive summaries.
Summarize the key points of the following text in exactly {num_sentences} sentences.
Focus on actionable insights and quantitative data.

Text: {text}

Summary:""",
    model="gemini-2.0-flash",
    temperature=0.2,
    metadata={"eval_score": 0.91, "author": "team-a"}
)

registry.register(
    task="classify",
    version="1.0",
    template="""Classify the following customer message into one of these categories:
{categories}

Message: {message}

Output only the category name.""",
    model="gemini-2.0-flash",
    temperature=0.0,
    metadata={"eval_score": 0.95}
)

# Activate production versions
registry.activate("summarize_v2.0")
registry.activate("classify_v1.0")

# Render a prompt
rendered = registry.render(
    "summarize",
    num_sentences=3,
    text="Vertex AI is Google Cloud's ML platform..."
)
print("\n--- Rendered Prompt ---")
print(rendered)

# List versions
print("\n--- All Summarize Versions ---")
for v in registry.list_versions("summarize"):
    print(f"  {v['id']}: score={v['metadata'].get('eval_score', 'N/A')}")

# Export to YAML
registry.export_yaml("prompts.yaml")

In [ ]:
# View the exported YAML
with open("prompts.yaml", "r") as f:
    print(f.read())

---
## 3. Automated Evaluation: LLM-as-Judge with Gemini

Use a strong model to evaluate the outputs of another model (or the same model with different prompts).

In [ ]:
# LLM-as-Judge evaluation framework
# This works with the Vertex AI SDK - shown with mock responses for demonstration

POINTWISE_JUDGE_PROMPT = """You are an expert evaluator. Rate the following AI response on these criteria:

1. **Relevance** (1-5): Does the response answer the question asked?
2. **Accuracy** (1-5): Is the information factually correct?
3. **Completeness** (1-5): Does it cover all important aspects?
4. **Clarity** (1-5): Is it well-organized and easy to understand?

Question: {question}
Response: {response}

Output ONLY valid JSON:
{{
  "relevance": <score>,
  "accuracy": <score>,
  "completeness": <score>,
  "clarity": <score>,
  "overall": <average>,
  "reasoning": "<brief explanation>"
}}"""

PAIRWISE_JUDGE_PROMPT = """You are an expert evaluator. Compare these two AI responses and determine which is better.

Question: {question}

Response A: {response_a}

Response B: {response_b}

Consider: relevance, accuracy, completeness, and clarity.

Output ONLY valid JSON:
{{
  "winner": "A" or "B" or "tie",
  "confidence": <0.0-1.0>,
  "reasoning": "<brief explanation>"
}}"""


class LLMJudge:
    """Automated evaluation using LLM-as-Judge pattern."""
    
    def __init__(self, judge_model_name: str = "gemini-2.0-pro"):
        self.judge_model_name = judge_model_name
        # In production: self.judge = GenerativeModel(judge_model_name)
    
    def pointwise_eval(self, question: str, response: str) -> Dict:
        """Score a single response on multiple criteria."""
        prompt = POINTWISE_JUDGE_PROMPT.format(
            question=question, response=response
        )
        
        # In production:
        # result = self.judge.generate_content(prompt)
        # return json.loads(result.text)
        
        # Mock response for demonstration
        print(f"[Judge] Evaluating response to: '{question[:50]}...'")
        return {
            "relevance": 4, "accuracy": 5, "completeness": 3,
            "clarity": 4, "overall": 4.0,
            "reasoning": "Good accuracy but missing some edge cases."
        }
    
    def pairwise_eval(self, question: str, response_a: str, response_b: str) -> Dict:
        """Compare two responses and pick the better one."""
        prompt = PAIRWISE_JUDGE_PROMPT.format(
            question=question, response_a=response_a, response_b=response_b
        )
        
        # In production:
        # result = self.judge.generate_content(prompt)
        # return json.loads(result.text)
        
        print(f"[Judge] Comparing two responses for: '{question[:50]}...'")
        return {
            "winner": "B", "confidence": 0.85,
            "reasoning": "Response B is more detailed and better structured."
        }
    
    def evaluate_batch(self, test_cases: List[Dict]) -> Dict:
        """Evaluate a batch of test cases and compute aggregate scores."""
        results = []
        for case in test_cases:
            result = self.pointwise_eval(case["question"], case["response"])
            results.append(result)
        
        # Compute aggregates
        avg_scores = {}
        for key in ["relevance", "accuracy", "completeness", "clarity", "overall"]:
            avg_scores[key] = sum(r[key] for r in results) / len(results)
        
        return {
            "individual_results": results,
            "aggregate_scores": avg_scores,
            "num_evaluated": len(results),
        }


# Demo usage
judge = LLMJudge()

# Pointwise evaluation
print("=== Pointwise Evaluation ===")
score = judge.pointwise_eval(
    question="What is Vertex AI?",
    response="Vertex AI is Google Cloud's unified ML platform for building and deploying ML models."
)
print(f"Scores: {score}\n")

# Pairwise evaluation
print("=== Pairwise Evaluation ===")
comparison = judge.pairwise_eval(
    question="How to deploy a model on Vertex AI?",
    response_a="Use the Vertex AI console to deploy.",
    response_b="Use the Vertex AI SDK: aiplatform.Model.deploy() specifying machine type, min replicas, and traffic split."
)
print(f"Result: {comparison}\n")

# Batch evaluation
print("=== Batch Evaluation ===")
test_cases = [
    {"question": "What is BigQuery?", "response": "BigQuery is a serverless data warehouse."},
    {"question": "What is Cloud Run?", "response": "Cloud Run is a serverless container platform."},
    {"question": "What is Pub/Sub?", "response": "Pub/Sub is a messaging service for event-driven systems."},
]
batch_result = judge.evaluate_batch(test_cases)
print(f"Aggregate scores: {batch_result['aggregate_scores']}")

---
## 4. RAG Pipeline: Embed, Retrieve, Generate

A complete RAG pipeline using Vertex AI embeddings and a simple in-memory vector store.

In [ ]:
import numpy as np

class SimpleVectorStore:
    """A minimal in-memory vector store for RAG demonstrations.
    
    In production, use Vertex AI Vector Search (Matching Engine)
    or a managed vector database like AlloyDB with pgvector.
    """
    
    def __init__(self, dimension: int = 768):
        self.dimension = dimension
        self.documents: List[Dict] = []
        self.embeddings: List[np.ndarray] = []
    
    def add_document(self, text: str, metadata: Optional[Dict] = None, embedding: Optional[np.ndarray] = None):
        """Add a document with its embedding."""
        if embedding is None:
            # Mock embedding - in production use Vertex AI text-embedding model
            embedding = self._mock_embed(text)
        
        self.documents.append({"text": text, "metadata": metadata or {}})
        self.embeddings.append(embedding)
    
    def search(self, query: str, top_k: int = 3, query_embedding: Optional[np.ndarray] = None) -> List[Dict]:
        """Find the most similar documents to a query."""
        if query_embedding is None:
            query_embedding = self._mock_embed(query)
        
        # Cosine similarity
        similarities = []
        for emb in self.embeddings:
            sim = np.dot(query_embedding, emb) / (
                np.linalg.norm(query_embedding) * np.linalg.norm(emb) + 1e-8
            )
            similarities.append(sim)
        
        # Get top-k indices
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        results = []
        for idx in top_indices:
            results.append({
                "text": self.documents[idx]["text"],
                "metadata": self.documents[idx]["metadata"],
                "score": float(similarities[idx]),
            })
        return results
    
    def _mock_embed(self, text: str) -> np.ndarray:
        """Generate a deterministic mock embedding from text.
        In production, use: vertexai.language_models.TextEmbeddingModel
        """
        np.random.seed(hash(text) % 2**32)
        return np.random.randn(self.dimension).astype(np.float32)


# Build a RAG knowledge base
store = SimpleVectorStore(dimension=768)

# Add documents about GCP services
documents = [
    {"text": "Vertex AI is Google Cloud's unified ML platform. It provides tools for data preparation, model training, evaluation, deployment, and monitoring. Key features include AutoML, custom training, Model Garden, and Vertex AI Studio.", "source": "gcp-docs"},
    {"text": "BigQuery ML lets you create and execute ML models using SQL queries. Supported models include linear regression, logistic regression, K-means clustering, matrix factorization, time-series forecasting, and deep neural networks.", "source": "gcp-docs"},
    {"text": "Cloud Run is a fully managed serverless platform for containerized applications. It automatically scales based on traffic, supports custom domains, and integrates with Cloud Build for CI/CD. You pay only for the resources you use.", "source": "gcp-docs"},
    {"text": "Vertex AI Pipelines is a serverless orchestration service for ML workflows. It supports both TFX and KFP (Kubeflow Pipelines) SDKs. Pipelines can include data preprocessing, training, evaluation, and deployment steps.", "source": "gcp-docs"},
    {"text": "Vertex AI Model Monitoring detects feature skew and drift in deployed models. It compares serving data distributions against training data baselines and triggers alerts when thresholds are exceeded.", "source": "gcp-docs"},
    {"text": "Gemini is Google's multimodal AI model family. Gemini Pro handles complex reasoning tasks, Gemini Flash is optimized for speed and cost, and Gemini Ultra handles the most demanding tasks. All are available through Vertex AI.", "source": "gcp-docs"},
]

for doc in documents:
    store.add_document(doc["text"], metadata={"source": doc["source"]})

print(f"Indexed {len(documents)} documents")

# Search
query = "How do I monitor ML models in production?"
results = store.search(query, top_k=3)

print(f"\nQuery: '{query}'")
print("\nTop results:")
for i, r in enumerate(results):
    print(f"\n  [{i+1}] Score: {r['score']:.4f}")
    print(f"      {r['text'][:100]}...")

In [ ]:
# Complete RAG pipeline: Retrieve + Generate

def rag_generate(query: str, store: SimpleVectorStore, top_k: int = 3) -> Dict:
    """Full RAG pipeline: retrieve relevant docs then generate an answer."""
    
    # Step 1: Retrieve
    retrieved = store.search(query, top_k=top_k)
    context = "\n\n".join([f"[Source {i+1}]: {r['text']}" for i, r in enumerate(retrieved)])
    
    # Step 2: Build prompt with context
    rag_prompt = f"""Answer the question using ONLY the provided context. If the context doesn't contain
the answer, say "I don't have enough information to answer this."

Context:
{context}

Question: {query}

Answer:"""
    
    # Step 3: Generate (mock - in production use Gemini)
    # response = model.generate_content(rag_prompt)
    
    return {
        "query": query,
        "retrieved_docs": retrieved,
        "prompt": rag_prompt,
        "prompt_length": len(rag_prompt),
        # "answer": response.text  # In production
    }


# Run RAG pipeline
result = rag_generate("How do I monitor ML models in production?", store)

print("=== RAG Pipeline Output ===")
print(f"\nQuery: {result['query']}")
print(f"Retrieved {len(result['retrieved_docs'])} documents")
print(f"Prompt length: {result['prompt_length']} characters")
print(f"\n--- Generated Prompt ---")
print(result['prompt'][:500] + "...")

---
## 5. Cost Tracking for GenAI API Calls

Track token usage and estimate costs for Gemini API calls.

In [ ]:
import pandas as pd
from datetime import datetime, timedelta

class GenAICostTracker:
    """Track and analyze costs for GenAI API calls.
    
    Pricing as of 2025 (per 1K tokens, may change):
    - Gemini 2.0 Flash: $0.0001 input / $0.0004 output
    - Gemini 2.0 Pro:   $0.00125 input / $0.005 output
    """
    
    PRICING = {
        "gemini-2.0-flash": {"input_per_1k": 0.0001, "output_per_1k": 0.0004},
        "gemini-2.0-pro":   {"input_per_1k": 0.00125, "output_per_1k": 0.005},
    }
    
    def __init__(self):
        self.requests: List[Dict] = []
    
    def log_request(self, model: str, input_tokens: int, output_tokens: int,
                    latency_ms: float = 0, metadata: Optional[Dict] = None):
        """Log a single API request."""
        pricing = self.PRICING.get(model, self.PRICING["gemini-2.0-flash"])
        input_cost = (input_tokens / 1000) * pricing["input_per_1k"]
        output_cost = (output_tokens / 1000) * pricing["output_per_1k"]
        
        self.requests.append({
            "timestamp": datetime.now().isoformat(),
            "model": model,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": input_tokens + output_tokens,
            "input_cost": input_cost,
            "output_cost": output_cost,
            "total_cost": input_cost + output_cost,
            "latency_ms": latency_ms,
            "metadata": metadata or {},
        })
    
    def summary(self) -> Dict:
        """Get a summary of all tracked requests."""
        if not self.requests:
            return {"message": "No requests tracked"}
        
        df = pd.DataFrame(self.requests)
        return {
            "total_requests": len(self.requests),
            "total_input_tokens": int(df["input_tokens"].sum()),
            "total_output_tokens": int(df["output_tokens"].sum()),
            "total_cost": round(df["total_cost"].sum(), 6),
            "avg_cost_per_request": round(df["total_cost"].mean(), 6),
            "avg_latency_ms": round(df["latency_ms"].mean(), 1),
            "cost_by_model": df.groupby("model")["total_cost"].sum().to_dict(),
        }
    
    def to_dataframe(self) -> pd.DataFrame:
        """Convert to DataFrame for analysis."""
        return pd.DataFrame(self.requests)


# Simulate a day of API usage
tracker = GenAICostTracker()

# Simulate various request types
import random
random.seed(42)

for i in range(50):
    model = random.choice(["gemini-2.0-flash", "gemini-2.0-flash", "gemini-2.0-pro"])  # 2:1 Flash:Pro
    input_tokens = random.randint(200, 3000)
    output_tokens = random.randint(50, 1000)
    latency = random.uniform(200, 2000) if model == "gemini-2.0-pro" else random.uniform(100, 500)
    
    tracker.log_request(
        model=model,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        latency_ms=latency,
        metadata={"task": random.choice(["summarize", "classify", "qa", "generate"])}
    )

# Print summary
summary = tracker.summary()
print("=== Cost Tracking Summary ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

# Show DataFrame
print("\n=== Sample Requests ===")
df = tracker.to_dataframe()
print(df[["model", "input_tokens", "output_tokens", "total_cost", "latency_ms"]].head(10).to_string(index=False))

---
## 6. Model Comparison Framework

Compare multiple models or configurations on the same evaluation dataset.

In [ ]:
class ModelComparison:
    """Framework for comparing multiple models/configurations.
    
    Use this to decide between:
    - Different models (Flash vs Pro)
    - Different prompt versions
    - Fine-tuned vs base model
    - RAG vs no-RAG
    """
    
    def __init__(self, test_cases: List[Dict]):
        self.test_cases = test_cases
        self.results: Dict[str, List[Dict]] = {}
    
    def add_model_results(self, model_name: str, responses: List[str], 
                          latencies: List[float], costs: List[float]):
        """Add results for a model configuration."""
        self.results[model_name] = []
        for i, case in enumerate(self.test_cases):
            self.results[model_name].append({
                "question": case["question"],
                "expected": case.get("expected", ""),
                "response": responses[i],
                "latency_ms": latencies[i],
                "cost": costs[i],
            })
    
    def compare(self) -> pd.DataFrame:
        """Generate a comparison report across all models."""
        summary = []
        for model_name, results in self.results.items():
            latencies = [r["latency_ms"] for r in results]
            costs = [r["cost"] for r in results]
            resp_lengths = [len(r["response"]) for r in results]
            
            summary.append({
                "Model": model_name,
                "Avg Latency (ms)": round(np.mean(latencies), 1),
                "P95 Latency (ms)": round(np.percentile(latencies, 95), 1),
                "Total Cost ($)": round(sum(costs), 6),
                "Avg Response Len": round(np.mean(resp_lengths)),
                "Num Tests": len(results),
            })
        
        return pd.DataFrame(summary)


# Define test cases
test_cases = [
    {"question": "What is Vertex AI?", "expected": "Google Cloud ML platform"},
    {"question": "How does AutoML work?", "expected": "Automated model training"},
    {"question": "What is model monitoring?", "expected": "Tracking model performance"},
    {"question": "Explain feature store.", "expected": "Centralized feature management"},
    {"question": "What is MLOps?", "expected": "ML operations practices"},
]

comparison = ModelComparison(test_cases)

# Simulate results from different configurations
random.seed(123)

# Model A: Gemini Flash (fast, cheap)
comparison.add_model_results(
    model_name="Gemini Flash",
    responses=[f"Short answer about {tc['question'][:20]}" for tc in test_cases],
    latencies=[random.uniform(100, 300) for _ in test_cases],
    costs=[random.uniform(0.0001, 0.0005) for _ in test_cases],
)

# Model B: Gemini Pro (slower, more detailed)
comparison.add_model_results(
    model_name="Gemini Pro",
    responses=[f"Detailed comprehensive answer about {tc['question'][:20]} with examples and references" for tc in test_cases],
    latencies=[random.uniform(500, 1500) for _ in test_cases],
    costs=[random.uniform(0.001, 0.005) for _ in test_cases],
)

# Model C: Fine-tuned Flash
comparison.add_model_results(
    model_name="Fine-tuned Flash",
    responses=[f"Domain-specific answer about {tc['question'][:20]} with GCP context" for tc in test_cases],
    latencies=[random.uniform(150, 400) for _ in test_cases],
    costs=[random.uniform(0.0002, 0.0008) for _ in test_cases],
)

# Display comparison
report = comparison.compare()
print("=== Model Comparison Report ===")
print(report.to_string(index=False))
print("\nRecommendation: Choose based on your quality/cost/latency tradeoffs.")
print("  - High volume, cost-sensitive -> Gemini Flash")
print("  - Quality-critical -> Gemini Pro")
print("  - Domain-specific + fast -> Fine-tuned Flash")

---
## Key Takeaways for the Exam

1. **GenAI MLOps differs from traditional MLOps** in evaluation (subjective), monitoring (quality drift + cost), and versioning (prompts + configs)
2. **Prompt versioning** is critical - treat prompts as code artifacts with version control
3. **Fine-tuning decision**: Use RAG for knowledge, fine-tuning for format/style. Start with prompting, add RAG, fine-tune last
4. **LLM-as-Judge** is the scalable evaluation approach - both pointwise and pairwise
5. **Cost monitoring** is unique to GenAI due to per-token pricing
6. **RAG operations** require managing chunking, embeddings, and vector indexes as operational concerns